# 02 — Deploy Qwen3-4B on Free Google Colab (T4 GPU) & Connect to Your Frontend

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU** (Free).
2. Set `ADAPTER_PATH` (leave `None` to test `v1.0-base` before training, or point to your saved LoRA folder for `v1.1-qlora`).
3. Run all cells — copy the `https://....trycloudflare.com` URL and paste it into your Middle Dashboard (`/dashboard`).

In [ ]:
# Cell 1: Install serving dependencies & cloudflared
!pip install -q fastapi uvicorn transformers accelerate bitsandbytes peft
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
# Cell 2: Mount Drive (if loading a trained LoRA adapter)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Load Qwen3-4B in 4-bit (+ optional LoRA adapter) on Free Colab T4 GPU
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-4B"
ADAPTER_PATH = None  # e.g. "/content/drive/MyDrive/my-custom-ai/adapters/v1.1-qlora"
MODEL_VERSION = "v1.0-base" if ADAPTER_PATH is None else "v1.1-qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

if ADAPTER_PATH:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print("Loaded custom LoRA adapter:", ADAPTER_PATH)

model.eval()
print(f"Ready! Active version: {MODEL_VERSION}")

In [ ]:
# Cell 4: Start FastAPI + Cloudflare Public Tunnel
import subprocess, re, time, threading
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Dict

api = FastAPI()

class ChatReq(BaseModel):
    prompt: str
    system_prompt: str = "You are a helpful, accurate AI assistant."
    history: List[Dict[str, str]] = []
    enable_thinking: bool = True

@api.get("/health")
def health():
    return {"status": "ok", "model": BASE_MODEL, "model_version": MODEL_VERSION}

@api.post("/chat")
def chat(req: ChatReq):
    messages = [{"role": "system", "content": req.system_prompt}]
    messages.extend(req.history)
    messages.append({"role": "user", "content": req.prompt})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=req.enable_thinking,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.6 if req.enable_thinking else 0.7,
            top_p=0.95,
            do_sample=True,
        )
    generated_ids = out[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return {"response": response_text, "model_version": MODEL_VERSION}

threading.Thread(target=lambda: uvicorn.run(api, host="0.0.0.0", port=8000), daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)
for line in proc.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n" + "="*60)
        print("PASTE THIS URL IN YOUR MIDDLE DASHBOARD (/dashboard):")
        print(match.group(0))
        print("="*60 + "\n")
        break